# Librerias 

In [20]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# Datos 

In [21]:
RANDOM_STATE = 42
# Cargar datos
df_ml = pd.read_csv(r"D:\\Datases_CD\\7mo\\Aprendizaje_Automatico\\Proyecto\\datos_preprocesados_finales_5y.csv")

# Asegurar tipo datetime y ordenar
df_ml["Date"] = pd.to_datetime(df_ml["Date"])
df_ml = df_ml.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Crear retorno futuro a 5 días por ticker fwd_ret_5d(t) = retorno acumulado de t a t+5 usando que ret_5d(t+5) = Close_{t+5} / Close_t - 1
df_ml["fwd_ret_5d"] = (
    df_ml
    .groupby("Ticker")["ret_5d"]   # tu ret_5d actual (trailing)
    .shift(-5)                     # lo movemos 5 días hacia arriba
)

df_ml.tail(2)


,Date,Close,Ticker,Return,ret_5d,ret_20d,vol_5d,vol_20d,ma_20,ma_60,dist_ma_20,dist_ma_60,rsi_10,stoch_k_10,spy_ret_20d,excess_ret_20d_vs_spy,categoria,fwd_ret_5d
144422,2025-11-21,2369.590088,^RUT,0.027973,-0.007805,-0.057244,0.01936,0.013301,2426.043994,2433.978333,-0.023270,-0.026454,38.217873,42.097025,-0.026903,-0.030341,Macro,NaN
144423,2025-11-24,2414.283447,^RUT,0.018861,0.031137,-0.042118,0.01791,0.014120,2420.736169,2434.776058,-0.002666,-0.008417,42.872611,71.275965,-0.024094,-0.018024,Macro,NaN


In [22]:
TARGET_COL = "fwd_ret_5d"

# columnas que queremos excluir de X
cols_excluir = ["Date", "Ticker", "categoria", TARGET_COL]
feature_cols = [c for c in df_ml.columns if c not in cols_excluir]

print("Target:", TARGET_COL)
print("Features usadas:", feature_cols)

Target: fwd_ret_5d
Features usadas: ['Close', 'Return', 'ret_5d', 'ret_20d', 'vol_5d', 'vol_20d', 'ma_20', 'ma_60', 'dist_ma_20', 'dist_ma_60', 'rsi_10', 'stoch_k_10', 'spy_ret_20d', 'excess_ret_20d_vs_spy']


# Preparacion de datos 

In [23]:
def preparar_datos_ticker(df, ticker, feature_cols, target_col="fwd_ret_5d", test_size_frac=0.2):
    # Todo el histórico del ticker ordenado
    df_ticker = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    if df_ticker.empty:
        raise ValueError(f"No hay datos para el ticker {ticker}")

    # Subconjunto con target disponible (ignorar las últimas 5 filas sin futuro)
    df_model = df_ticker[df_ticker[target_col].notna()].copy()

    if df_model.empty:
        raise ValueError(f"No hay filas con target para {ticker}")

    X = df_model[feature_cols].values
    y = df_model[target_col].values

    n = len(df_model)
    test_size = max(1, int(np.floor(n * test_size_frac)))
    train_size = n - test_size

    if train_size < 10:
        raise ValueError(f"Muy pocos datos de entrenamiento para {ticker}: {train_size} filas útiles")

    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    return X_train, X_test, y_train, y_test, df_ticker, df_model

Funcion para RMSE , para las paqueterias de sklearn 

In [24]:
try:
    from sklearn.metrics import root_mean_squared_error
    def rmse(y_true, y_pred):
        return root_mean_squared_error(y_true, y_pred)
except ImportError:
    from sklearn.metrics import mean_squared_error
    def rmse(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)


# Optimizacion Bayesiana 

In [25]:
def bayes_opt_rf_time_series(X_train, y_train, X_test, y_test,
                             n_splits=5,
                             n_iter=30):
    """
    Optimización bayesiana de hiperparámetros para RandomForest
    usando TimeSeriesSplit en el conjunto de entrenamiento.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search_spaces = {
        "n_estimators": Integer(50, 400),
        "max_depth": Integer(3, 20),
        "max_features": Real(0.3, 1.0),
        "min_samples_leaf": Integer(1, 20),
        "min_samples_split": Integer(2, 30)
    }

    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=n_iter,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0
    )

    opt.fit(X_train, y_train)

    best_model = opt.best_estimator_
    best_params = opt.best_params_
    cv_mae_mean = -opt.best_score_
    cv_mae_std = opt.cv_results_["std_test_score"][opt.best_index_]

    # Evaluación en el test (último tramo temporal con futuro conocido)
    y_pred = best_model.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = rmse(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    return {
        "modelo_train": best_model,  # entrenado solo con X_train, y_train
        "best_params": best_params,
        "cv_mae_mean": cv_mae_mean,
        "cv_mae_std": cv_mae_std,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    }


# Modelo de Bayes para todo los Tickers

In [26]:
def correr_rf_bayes_en_todos_los_tickers(df,
                                         feature_cols,
                                         target_col="fwd_ret_5d",
                                         test_size_frac=0.2,
                                         n_splits=5,
                                         n_iter=30):
    resultados = {}
    tickers = df["Ticker"].unique()
    print(f"Voy a entrenar RF (BayesSearch) para {len(tickers)} tickers")

    for ticker in tickers:
        try:
            X_train, X_test, y_train, y_test, df_ticker, df_model = preparar_datos_ticker(
                df,
                ticker=ticker,
                feature_cols=feature_cols,
                target_col=target_col,
                test_size_frac=test_size_frac
            )

            res = bayes_opt_rf_time_series(
                X_train, y_train, X_test, y_test,
                n_splits=n_splits,
                n_iter=n_iter
            )

            resultados[ticker] = res

            print(
                f"[RF Bayes] {ticker}: "
                f"CV_MAE={res['cv_mae_mean']:.5f}, "
                f"Test_MAE={res['test_mae']:.5f}, "
                f"Test_R2={res['test_r2']:.4f}, "
                f"Best={res['best_params']}"
            )

        except ValueError as e:
            print(f"Saltando {ticker}: {e}")

    return resultados


In [27]:
resultados_rf_bayes = correr_rf_bayes_en_todos_los_tickers(
    df_ml,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    test_size_frac=0.2,
    n_splits=5,
    n_iter=30   # puedes bajar a 15-20 si tarda mucho
)

Voy a entrenar RF (BayesSearch) para 116 tickers
[RF Bayes] AAPL: CV_MAE=0.02986, Test_MAE=0.03510, Test_R2=0.0007, Best=OrderedDict({'max_depth': 3, 'max_features': 0.30095584379786405, 'min_samples_leaf': 3, 'min_samples_split': 30, 'n_estimators': 88})


KeyboardInterrupt: 